## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# Bubble-tea quarterly revenue: seasonal average indices

**Seasonal Average Method** · a worked example from the StatNex teaching guide

**The story.** A bubble-tea chain's quarterly revenue from 2020 to 2024. Summer (Q3) is the best quarter and January to March (Q1) the weakest in every year, and the business grew every year.

**The question.** How much stronger is a typical summer quarter than an average quarter?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/period_average.csv` (20 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Year / period,Sales ($000)
2020-1Q,792.0
2020-2Q,1132.0
2020-3Q,1359.0
2020-4Q,852.0
2021-1Q,915.0
2021-2Q,1216.0
2021-3Q,1503.0
2021-4Q,960.0
2022-1Q,1005.0
2022-2Q,1385.0
2022-3Q,1732.0
2022-4Q,1083.0
2023-1Q,1075.0
2023-2Q,1600.0
2023-3Q,1801.0
2023-4Q,1140.0
2024-1Q,1184.0
2024-2Q,1522.0
2024-3Q,1861.0
2024-4Q,1247.0
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
# One column in time order, cut into cycles of period_length observations and laid
# side by side: one row per period, one column per cycle. A last, incomplete cycle
# leaves its missing periods empty.
values = data["Sales ($000)"].to_numpy(float)
period_length = 4
n_cycles = int(np.ceil(len(values) / period_length))
padded = np.full(n_cycles * period_length, np.nan)
padded[:len(values)] = values
cycles = [f"Cycle {c + 1}" for c in range(n_cycles)]
table = pd.DataFrame(padded.reshape(n_cycles, period_length).T, columns=cycles)
table.index = pd.RangeIndex(1, len(table) + 1, name="Period")

## Period averages and the seasonal index

In [ ]:
# Average each period across the cycles (skipping a missing one), then divide by the
# mean of those averages: an index above 1 marks a period above the yearly level, so
# 1.30 reads "30% above an average period". No library function computes this simple
# index, so it is written out; it is two lines of averages.
# The method assumes the series has no trend. In a growing series the later periods
# of every cycle sit higher on the trend, so their indices come out too high and the
# early periods' too low. Methods that take the trend out first (the link relative
# method, or the ratio to a centred moving average in statsmodels' seasonal_decompose)
# avoid that.
table["Period average"] = table[cycles].mean(axis=1)
table["Period index"] = table["Period average"] / table["Period average"].mean()
print(table)

## What the result says

The seasonal average method takes the mean revenue of each quarter over the five years, the period average. It then divides it by the mean of all quarters. The result is the period index, where 1 means an average quarter.

The period average is 1651.2 for the third quarter (summer) and 994.2 for the first, in \$000. The period index is 1.302 for the third quarter, 1.081 for the second, 0.833 for the fourth and 0.784 for the first. So a typical summer quarter brings in almost a third more than an average quarter, and the first quarter more than a fifth less.

Read these figures with care. First-quarter revenue grew from 792 in the first year to 1184 in the fifth, and this method leaves that growth in. Part of each index is growth, not season.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.